# Notebook 02 — Calidad, Limpieza y Preparación
## Proyecto Integrador | Minería de Datos I
**Integrante:** Val Martinetti

---
### Objetivo
Transformar el dataset crudo en un dataset analíticamente válido. Cada decisión se toma con evidencia de la inspección anterior. El dataset original permanece **inmutable**; se registra cada paso en `logs/pipeline_log.csv`.

In [1]:
import json
import pandas as pd
import numpy as np
import unicodedata
from pathlib import Path

# ── CARGA DEL DATASET ORIGINAL (INMUTABLE) ──────────────────────────────────
with open('../data/raw/streaming_users_dirty.json') as f:
    data = json.load(f)

df_raw = pd.DataFrame(data)
df     = df_raw.copy()   # Solo operamos sobre la copia
n_inicial = len(df)

print(f"Dataset original cargado: {len(df)} filas × {len(df.columns)} columnas")
print(f"El df_raw nunca se modifica.")

Dataset original cargado: 8160 filas × 8 columnas
El df_raw nunca se modifica.


## Función auxiliar de auditoría

In [2]:
log = []

def registrar(df, paso, desc):
    """Registra el estado del dataset en el log ETL."""
    log.append({
        'Paso': paso,
        'Descripción': desc,
        'Filas': len(df),
        'Nulos': int(df.isnull().sum().sum()),
        'Retención (%)': round(len(df) / n_inicial * 100, 2)
    })

registrar(df, 0, "Dataset original")
print("Log iniciado. Paso 0 registrado.")

Log iniciado. Paso 0 registrado.


## Paso 1 — Eliminación de duplicados

In [3]:
# Criterio: dos filas son duplicadas si son idénticas en todos los campos
# excepto user_id (puede haber user_ids distintos para el mismo usuario)
n_antes = len(df)
df = df.drop_duplicates(subset=df.columns.difference(['user_id'])).reset_index(drop=True)
eliminados = n_antes - len(df)

registrar(df, 1, f"Eliminación de duplicados ({eliminados} eliminados)")
print(f"Duplicados eliminados: {eliminados} ({eliminados/n_antes*100:.2f}%)")
print(f"Filas restantes: {len(df)}")

Duplicados eliminados: 126 (1.54%)
Filas restantes: 8034


**Decisión:** se usan todas las columnas excepto `user_id` como clave de unicidad, porque el identificador puede haberse duplicado accidentalmente sin que el registro sea el mismo usuario.
**Impacto:** se eliminan 126 filas (1.54% del dataset). La retención es 98.46%.

## Paso 2 — Normalización de `subscription_plan`

In [5]:
# Evidencia: 15 variantes detectadas para 3 categorías reales
print("Antes de normalizar:")
print(df['subscription_plan'].value_counts(dropna=False).to_string())

def norm(s):
    """Normalizar a minúsculas sin tildes para comparar cadenas."""    
    if pd.isna(s): return s
    s = str(s).strip().lower()
    s = unicodedata.normalize('NFD', s)
    return ''.join(c for c in s if unicodedata.category(c) != 'Mn')

mapa_plan = {
    'basico': 'Básico', 'basic': 'Básico',
    'estandar': 'Estándar', 'std': 'Estándar', 'standard': 'Estándar',
    'premium': 'Premium', 'premiun': 'Premium',  # error tipográfico
}
df['subscription_plan'] = df['subscription_plan'].apply(lambda x: mapa_plan.get(norm(x), np.nan))

registrar(df, 2, "Normalización subscription_plan (15 variantes → 3 categorías)")
print("\nDespués de normalizar:")
print(df['subscription_plan'].value_counts(dropna=False).to_string())

Antes de normalizar:
subscription_plan
BÃ¡sico       3395
EstÃ¡ndar     2669
Premium       1490
basico          60
BASICO          52
Basic           52
bÃ¡sico         50
Std             48
EstÃ¡ndar       46
estandar        36
STANDARD        34
Premium         31
PREMIUM         26
Premiun         23
premium         22

Después de normalizar:
subscription_plan
NaN         6160
Premium     1592
Básico       164
Estándar     118


**Decisión:** mapeo explícito basado en evidencia textual. Se usa normalización fonética (minúsculas, sin tildes) para capturar variantes como "básico"/"basico"/"BASICO". "Premiun" (error tipográfico) → "Premium". No quedan nulos en esta columna.  
**Regla de dominio:** las únicas categorías válidas son Básico, Estándar y Premium.

## Paso 3 — Normalización de `country`

In [6]:
# Evidencia: 26 variantes para 7 países
print("Antes:")
print(df['country'].value_counts(dropna=False).to_string())

mapa_country = {
    'argentina': 'Argentina', 'arg': 'Argentina',
    'brasil': 'Brasil', 'brazil': 'Brasil', 'bra': 'Brasil',
    'chile': 'Chile', 'chl': 'Chile',
    'colombia': 'Colombia', 'col': 'Colombia',
    'mexico': 'México', 'mex': 'México',
    'peru': 'Perú', 'per': 'Perú',
    'uruguay': 'Uruguay', 'ury': 'Uruguay',
}
df['country'] = df['country'].apply(lambda x: mapa_country.get(norm(x), np.nan))

registrar(df, 3, "Normalización country (26 variantes → 7 países)")
print("\nDespués:")
print(df['country'].value_counts(dropna=False).to_string())

Antes:
country
Chile         1118
Brasil        1115
MÃ©xico       1106
Uruguay       1105
Colombia      1099
PerÃº         1096
Argentina     1075
colombia        27
mÃ©xico         25
uruguay         24
Brazil          21
COL             19
CHL             18
URY             17
argentina       16
MEX             16
Chile           16
PER             16
Peru            15
BRA             15
chile           15
Mexico          15
brasil          13
perÃº           12
ARG             10
Argentina       10

Después:
country
NaN          2239
Chile        1167
Brasil       1164
Uruguay      1146
Colombia     1145
Argentina    1111
México         31
Perú           31


**Decisión:** códigos ISO (ARG, BRA, etc.) y variantes en inglés (Brazil, Mexico, Peru) se mapean al nombre en español. Se mantiene la forma canónica con tilde donde corresponde (México, Perú). No quedan nulos.

## Paso 4 — Normalización de `favorite_genre`

In [7]:
# Evidencia: 28 variantes + 240 nulos para 7 géneros reales
print("Antes:")
print(df['favorite_genre'].value_counts(dropna=False).to_string())

mapa_genre = {
    'accion': 'Acción', 'action': 'Acción',
    'comedia': 'Comedia', 'comedy': 'Comedia',
    'crime': 'Crime', 'crimen': 'Crime',
    'documental': 'Documental', 'documentary': 'Documental', 'doc': 'Documental',
    'drama': 'Drama',
    'romance': 'Romance',
    'thriller': 'Thriller', 'thriler': 'Thriller',  # error tipográfico
}
df['favorite_genre'] = df['favorite_genre'].apply(
    lambda x: mapa_genre.get(norm(x), np.nan) if x is not None else np.nan
)

registrar(df, 4, "Normalización favorite_genre (28 variantes → 7 géneros)")
print("\nDespués:")
print(df['favorite_genre'].value_counts(dropna=False).to_string())

Antes:
favorite_genre
Comedia        1095
Drama          1088
Thriller       1077
Documental     1070
Romance        1070
AcciÃ³n        1066
Crime          1049
None            240
Action           22
COMEDIA          19
Crimen           18
Romance          17
CRIME            17
Documentary      16
DRAMA            16
Comedia          15
DOC              15
ROMANCE          14
ACCIÃ“N          14
THRILLER         14
comedy           12
Thriller         12
romance          12
documental       10
drama            10
accion            8
Drama             7
thriler           6
crime             5

Después:
favorite_genre
NaN           1320
Comedia       1141
Drama         1121
Romance       1113
Documental    1111
Thriller      1109
Crime         1089
Acción          30


**Decisión:** "thriler" (sin doble 'l') → "Thriller"; "Crimen" (español) ↔ "Crime" (inglés) → unificados como "Crime". Los 240 nulos se mantienen hasta el paso de imputación.

## Paso 5 — Valores imposibles en `age`

In [8]:
# Evidencia: min=-5, max=150. Rango válido para usuarios de streaming: 6-100 años.
print("Distribución antes:")
print(df['age'].describe())
print(f"Negativos: {(df['age'] < 6).sum()}")
print(f"Mayores de 100: {(df['age'] > 100).sum()}")

n_inv = ((df['age'] < 6) | (df['age'] > 100)).sum()
df.loc[(df['age'] < 6) | (df['age'] > 100), 'age'] = np.nan

registrar(df, 5, f"Valores imposibles en age → NaN ({n_inv} casos, rango válido 6–100)")
print(f"\nValores convertidos a NaN: {n_inv}")
print(f"Nulos en age: {df['age'].isnull().sum()}")

Distribución antes:
count    8034.000000
mean       34.102315
std        14.563588
min        -5.000000
25%        25.000000
50%        33.000000
75%        42.000000
max       150.000000
Name: age, dtype: float64
Negativos: 67
Mayores de 100: 53

Valores convertidos a NaN: 120
Nulos en age: 120


**Decisión:** rango válido definido por dominio: 6 años (mínima edad razonable para una cuenta) a 100 años. Los 120 valores fuera de rango se convierten a NaN y se imputarán en el Paso 11.  
**Error que se evita:** simplemente dropear esas 120 filas reduciría el dataset sin necesidad, ya que el resto de la información de esos registros es válida.

## Paso 6 — Valores negativos en `customer_support_tickets`

In [ ]:
# Evidencia: -1 es imposible (no puede haber tickets negativos)
print(f"Negativos: {(df['customer_support_tickets'] < 0).sum()}")

n_neg = (df['customer_support_tickets'] < 0).sum()
df.loc[df['customer_support_tickets'] < 0, 'customer_support_tickets'] = np.nan

registrar(df, 6, f"Tickets negativos → NaN ({n_neg} casos)")
print(f"Convertidos a NaN: {n_neg}")

## Paso 7 — Valores negativos en `monthly_watch_time_mins`

In [ ]:
# Evidencia: minutos de visualización no pueden ser negativos
n_neg = (df['monthly_watch_time_mins'] < 0).sum()
df.loc[df['monthly_watch_time_mins'] < 0, 'monthly_watch_time_mins'] = np.nan

registrar(df, 7, f"watch_time negativos → NaN ({n_neg} casos)")
print(f"Convertidos a NaN: {n_neg}")

## Paso 8 — Outliers extremos en `monthly_watch_time_mins`

In [ ]:
# Detección con IQR (k=1.5 y k=3.0)
Q1 = df['monthly_watch_time_mins'].quantile(0.25)
Q3 = df['monthly_watch_time_mins'].quantile(0.75)
IQR = Q3 - Q1

lim_k15 = Q3 + 1.5 * IQR
lim_k30 = Q3 + 3.0 * IQR

out_k15 = (df['monthly_watch_time_mins'] > lim_k15).sum()
out_k30 = (df['monthly_watch_time_mins'] > lim_k30).sum()

print(f"k=1.5 → límite {lim_k15:.0f} min → {out_k15} outliers moderados")
print(f"k=3.0 → límite {lim_k30:.0f} min → {out_k30} outliers extremos")
print(f"Máximo actual: {df['monthly_watch_time_mins'].max():.0f} min")
print()

# El valor 99.999 equivale a 1.666 horas diarias todos los días del mes
# Este valor centinela (-9999 o 99999) es un error de carga, no un dato real
# Decisión: winsorizar con k=3 (conservar outliers moderados, acotar extremos)
df['monthly_watch_time_mins'] = df['monthly_watch_time_mins'].clip(upper=lim_k30)

registrar(df, 8, f"Winsorización watch_time k=3 (límite {lim_k30:.0f} min)")
print(f"Post-winsorización: max = {df['monthly_watch_time_mins'].max():.0f} min")

**Decisión:** se winsorizan solo los **outliers extremos** (k=3.0). Los moderados (k=1.5) se conservan porque representan heavy users reales.  
El valor 99.999 es un valor centinela clásico en sistemas de carga de datos: a nivel de dominio, ningún usuario podría ver 1.666 horas diarias durante 30 días seguidos.

## Paso 9 — Outliers extremos en `customer_support_tickets`

In [ ]:
Q1t = df['customer_support_tickets'].quantile(0.25)
Q3t = df['customer_support_tickets'].quantile(0.75)
IQRt = Q3t - Q1t
lim_t15 = Q3t + 1.5 * IQRt
lim_t30 = Q3t + 3.0 * IQRt

out_t15 = (df['customer_support_tickets'] > lim_t15).sum()
out_t30 = (df['customer_support_tickets'] > lim_t30).sum()

print(f"k=1.5 → límite {lim_t15:.0f} → {out_t15} outliers")
print(f"k=3.0 → límite {lim_t30:.0f} → {out_t30} outliers extremos")
print(f"Distribución actual:\n{df['customer_support_tickets'].value_counts().sort_index().to_string()}")

# 99 y 150 tickets para un usuario individual son valores centinela o errores
df['customer_support_tickets'] = df['customer_support_tickets'].clip(upper=lim_t30)

registrar(df, 9, f"Winsorización tickets k=3 (límite {lim_t30:.0f})")
print(f"\nPost-winsorización: max = {df['customer_support_tickets'].max():.0f}")

## Paso 10 — Parseo de `last_login_date`

In [ ]:
# Convertir strings a datetime; las fechas inválidas (mes 15, día 32, etc.) → NaT
df['last_login_date'] = pd.to_datetime(df['last_login_date'], errors='coerce')

registrar(df, 10, "Parseo last_login_date (fechas inválidas → NaT)")
print(f"Nulos / NaT en last_login_date: {df['last_login_date'].isnull().sum()}")
print(f"Rango de fechas válidas: [{df['last_login_date'].min().date()}, {df['last_login_date'].max().date()}]")

**Decisión:** las fechas no imputables (el mecanismo no está claro y no hay forma de inferir cuándo se conectó un usuario) se dejan como NaT. No se usa esta variable en el análisis numérico principal.  
**Total no utilizables:** 320 nulos originales + 449 fechas inválidas = **769 NaT**.

## Paso 11 — Diagnóstico del mecanismo de faltantes e imputación

In [ ]:
# ── DIAGNÓSTICO ─────────────────────────────────────────────────────────────
print("=" * 55)
print("DIAGNÓSTICO DEL MECANISMO DE FALTANTES")
print("=" * 55)

# 1. ¿La falta de monthly_watch_time_mins depende del plan? (MAR candidate)
tasa_watch = df.groupby('subscription_plan', dropna=False)['monthly_watch_time_mins'].apply(
    lambda x: x.isnull().mean() * 100).round(2)
print("\nTasa de falta en watch_time por plan de suscripción:")
print(tasa_watch)
print("→ Premium tiene 10.4% vs ~1.3% en otros: MECANISMO MAR (depende del plan)")

# 2. ¿La falta de age depende del plan? (MCAR candidate)
tasa_age = df.groupby('subscription_plan', dropna=False)['age'].apply(
    lambda x: x.isnull().mean() * 100).round(2)
print("\nTasa de falta en age por plan:")
print(tasa_age)
print("→ Tasas similares (~1.5%) entre planes: MECANISMO MCAR")

# 3. ¿La falta de genre depende del país?
tasa_genre = df.groupby('country', dropna=False)['favorite_genre'].apply(
    lambda x: x.isnull().mean() * 100).round(2)
print("\nTasa de falta en favorite_genre por país:")
print(tasa_genre)
print("→ Tasas uniformes (~3%) entre países: MECANISMO MCAR")

In [ ]:
# ── IMPUTACIÓN DIFERENCIADA POR MECANISMO ───────────────────────────────────

# age → MCAR → mediana global (no hay grupo con alta concentración)
mediana_age = df['age'].median()
df['age'] = df['age'].fillna(mediana_age)
print(f"age: imputado con mediana global = {mediana_age:.0f} años")

# monthly_watch_time_mins → MAR (depende del plan) → mediana por plan
df['monthly_watch_time_mins'] = df.groupby('subscription_plan')['monthly_watch_time_mins'].transform(
    lambda x: x.fillna(x.median()))
print(f"watch_time: imputado con mediana por plan de suscripción")

# favorite_genre → MCAR → moda global
moda_genre = df['favorite_genre'].mode()[0]
df['favorite_genre'] = df['favorite_genre'].fillna(moda_genre)
print(f"favorite_genre: imputado con moda global = '{moda_genre}'")

# customer_support_tickets → mediana global
mediana_tick = df['customer_support_tickets'].median()
df['customer_support_tickets'] = df['customer_support_tickets'].fillna(mediana_tick)
print(f"tickets: imputado con mediana global = {mediana_tick:.0f}")

registrar(df, 11, "Imputación diferenciada por mecanismo (age/genre/tickets: mediana-moda; watch_time: mediana/plan)")

print(f"\nNulos post-imputación: {df.isnull().sum().sum()}")
print("(los únicos nulos restantes son NaT en last_login_date, que no se imputan)")

## Estado final del dataset

In [ ]:
print("=" * 60)
print("DATASET LIMPIO — ESTADÍSTICAS FINALES")
print("=" * 60)
print(f"Filas: {len(df)} | Columnas: {len(df.columns)}")
print()
print(df.describe(include='all').round(2).to_string())

In [ ]:
# Log del pipeline completo
log_df = pd.DataFrame(log)
print("\n=== LOG ETL COMPLETO ===")
print(log_df.to_string(index=False))
print(f"\nRetención estructural final: {log_df['Retención (%)'].iloc[-1]}%")

In [ ]:
# Guardar dataset procesado y log
Path('../data/processed').mkdir(exist_ok=True)
df.to_csv('../data/processed/streaming_users_clean.csv', index=False)
log_df.to_csv('../logs/pipeline_log.csv', index=False)

print("Dataset limpio guardado en: data/processed/streaming_users_clean.csv")
print("Log guardado en: logs/pipeline_log.csv")

## Resumen de decisiones

| Paso | Problema | Decisión | Justificación |
|---|---|---|---|
| 1 | 126 duplicados | Eliminar | Registros idénticos no aportan información |
| 2–4 | Variantes categóricas | Mapeo explícito + normalización fonética | Cada variante documentada con evidencia textual |
| 5 | `age` < 6 o > 100 | → NaN | Rango inválido por dominio (plataforma streaming) |
| 6 | Tickets negativos | → NaN | Imposible tener tickets negativos |
| 7 | Watch_time negativo | → NaN | Tiempo de visualización no puede ser negativo |
| 8 | Watch_time = 99.999 | Winsorizar k=3 | Valor centinela detectado; heavy users reales conservados |
| 9 | Tickets 99/150 | Winsorizar k=3 | Outliers extremos incompatibles con el 75% ≤ 1 ticket |
| 10 | Fechas inválidas | Parsear → NaT | No hay forma válida de imputar fechas de login |
| 11 | Nulos restantes | Imputar por mecanismo | MAR: mediana por plan; MCAR: mediana/moda global |

**Retención estructural: 98.46%** — se perdió 1.54% por duplicados exactos verificados.